## PVA 2 - Coding Kata - CV
Procedere:

1. Arbeitet in Zweier-Teams zur Lösung dieser Aufgabe
2. Abgabe der Notebooks auf Moodle
3. Verwendet OpenCV für die Lösung dieser Aufgabe


__Aufgabe 1__

Folgende Dateien entsprechen den Kanälen eines Landsat TM Satellitenbildes
- Landsat8_Blau.tif
- Landsat8_Gruen.tif
- Landsat8_Rot.tif
- Landsat8_NIR.tif

1. Finde Eigenschaften des Landsat 8 heraus
  - Wie ist die Bahn des Landsat 8 Satelliten
  - Welche Bildkanäle mit welcher Auflösung liefert der Landsat 8 Satellit
2. Stelle den blauen Kanal des Landsat8 Bildes dar
    - Welche Eigenschaften hat dieser Kanal (.shape)
    - Warum ist er so dunkel?
3. Erhöhe den Kontrast dieses Kanals mittels Histogrammaufspreizung


In [ ]:
uv add rasterio opencv-python numpy matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 64.9 MB/s eta 0:00:00


In [4]:
import rasterio
import numpy as np
import os

# Pfade zu den hochgeladenen TIFF-Dateien
files = {
    'Blau': 'Landsat8_Blau.tif',
    'Gruen': 'Landsat8_Gruen.tif',
    'Rot': 'Landsat8_Rot.tif',
    'NIR': 'Landsat8_NIR.tif'
}

# Eigenschaften der TIFF-Dateien speichern
band_info = {}

for band_name, file_path in files.items():
    path = os.path.join(os.getcwd(), "PVA2", "Daten", file_path)
    
    with rasterio.open(path) as src:
        band_data = src.read(1)  # lies das erste Band
        width = src.width
        height = src.height
        crs = src.crs  # Koordinatensystem
        transform = src.transform  # Georeferenzierung

        band_info[band_name] = {
            'Width': width,
            'Height': height,
            'CRS': crs,
            'Transform': transform,
            'Data Type': src.dtypes[0]
        }

# Ausgabe der gesammelten Informationen
for band_name, info in band_info.items():
    print(f"{band_name} Band:")
    for key, value in info.items():
        print(f"  {key}: {value}")


Blau Band:
  Width: 1096
  Height: 878
  CRS: EPSG:32632
  Transform: | 30.00, 0.00, 442935.00|
| 0.00,-30.00, 5267595.00|
| 0.00, 0.00, 1.00|
  Data Type: uint16
Gruen Band:
  Width: 1096
  Height: 878
  CRS: EPSG:32632
  Transform: | 30.00, 0.00, 442935.00|
| 0.00,-30.00, 5267595.00|
| 0.00, 0.00, 1.00|
  Data Type: uint16
Rot Band:
  Width: 1096
  Height: 878
  CRS: EPSG:32632
  Transform: | 30.00, 0.00, 442935.00|
| 0.00,-30.00, 5267595.00|
| 0.00, 0.00, 1.00|
  Data Type: uint16
NIR Band:
  Width: 1096
  Height: 878
  CRS: EPSG:32632
  Transform: | 30.00, 0.00, 442935.00|
| 0.00,-30.00, 5267595.00|
| 0.00, 0.00, 1.00|
  Data Type: uint16


In [14]:
import cv2
import matplotlib.pyplot as plt
# Eigenschaften der TIFF-Dateien speichern
bands = {}

for band_name, file_path in files.items():
    path = os.path.join(os.getcwd(), "PVA2", "Daten", file_path)

    image = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if image is None:
        raise FileNotFoundError(f"Image could not be found: {path}")
    bands[band_name] = image


blue = bands["Blau"]

print("Shape:", blue.shape)
print("Datentyp:", blue.dtype)
print("Minimum / Maximum:", blue.min(), blue.max())

def stretch(image):
    # Lineare Histogrammaufspreizung auf 0–255
    return cv2.normalize(
        image,
        None,
        alpha=0,
        beta=255,
        norm_type=cv2.NORM_MINMAX,
        dtype=cv2.CV_8U,
    )

blue_stretched = stretch(blue)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(blue, cmap="gray", vmin=0, vmax=65535)
axes[0].set_title("Blauer Kanal: ursprünglicher Wertebereich")

axes[1].imshow(blue_stretched, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Blauer Kanal: Histogrammaufspreizung")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) encountered
[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34736 (0x87b0) encountered
[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34737 (0x87b1) encountered
[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33550 (0x830e) encountered
[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 33922 (0x8482) encountered
[ WARN:0@197.430] global grfmt_tiff.cpp:123 TIFF_Warning TIFFReadDirectory: Unknown field with tag 34735 (0x87af) enco

Shape: (878, 1096)
Datentyp: uint16
Minimum / Maximum: 0 56914


__Aufgabe 2__

1. Erstelle mittels opencv ein Echtfarbenbild (R=Rot, G=Gruen, B=Blau) des Landsatbildes und stelle es dar.

2. Erstelle mittels opencv ein Faslchfarbenbild (R=NIR, G=Rot, B=Gruen) des Landsatbildes und stelle es dar.


In [15]:
def stretch_percentile(image):
    low, high = np.percentile(image, (2, 98))
    clipped = np.clip(image.astype(np.float32), low, high)
    return stretch(clipped)


In [16]:
display_bands = {
    name: stretch_percentile(image)
    for name, image in bands.items()
}

# Matplotlib expects RGB order.
true_color = cv2.merge([
    display_bands["Rot"],    # R
    display_bands["Gruen"],  # G
    display_bands["Blau"],   # B
])

false_color = cv2.merge([
    display_bands["NIR"],    # R
    display_bands["Rot"],    # G
    display_bands["Gruen"],  # B
])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(true_color)
axes[0].set_title("Echtfarben: R=Rot, G=Grün, B=Blau")

axes[1].imshow(false_color)
axes[1].set_title("Falschfarben: R=NIR, G=Rot, B=Grün")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()


__Aufgabe 3__

Image augmentation - um die Menge der Bilddatensätze zu erhöhen kann man die Trainingsbilder noch bearbeiten mit folgenden geometrischen Transformationen:
- Translation
- Rotation
- Skalierung
- Scherung (Shearing)
- Horzontale (oder vertikale) Spiegelungen

Implementiere eine Funktion, die neue Bilder erzeugt durch Rotation und Spiegelung, wobei für die Rotation ein Winkel und Schritte angegeben werden soll (z.B. von 30 -40 in Zwei-Grad-Schritten).


In [17]:
import cv2

def augment_images(image, start_angle=30, end_angle=40, step=2):
    """Generate rotations and horizontal mirrors, keeping the image size."""
    if step <= 0:
        raise ValueError("step must be positive")
    if start_angle > end_angle:
        raise ValueError("start_angle must be <= end_angle")

    height, width = image.shape[:2]
    center = ((width - 1) / 2, (height - 1) / 2)
    augmented = {}

    # Includes end_angle if it falls on a step.
    for angle in range(start_angle, end_angle + 1, step):
        matrix = cv2.getRotationMatrix2D(
            center,
            angle,
            scale=1.0,
        )

        rotated = cv2.warpAffine(
            image,
            matrix,
            (width, height),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT_101,
        )

        augmented[f"rotation_{angle}"] = rotated
        augmented[f"rotation_{angle}_flipped"] = cv2.flip(rotated, 1)

    return augmented


In [18]:
import matplotlib.pyplot as plt

images = augment_images(
    true_color,
    start_angle=30,
    end_angle=40,
    step=2,
)

fig, axes = plt.subplots(
    len(images) // 2, 2,
    figsize=(10, 3 * (len(images) // 2)),
)

for ax, (name, image) in zip(axes.flat, images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()


__Aufgabe 4__

Erstelle ein trainiertes Neuronales Netz für das Spiel "Schere, Stein, Papier". Verwende dazu folgenden Link zu [TeachableMachine](https://teachablemachine.withgoogle.com/train/image) und erstelle Trainingsdaten via Webcam. Exportiere anschliessend dein trainiertes Model als Tensorflow/Keras-Modell und erstelle ein neues Notebook, wo du dieses Modell dann mit einem Testbild verwendest.


Da ich mich nicht zwingend selbst aufnehmen möchte, habe ich [hier](https://platform.ultralytics.com/kjell-juergens/datasets/rock-paper-scissors) alternative Daten gefunden und trainiere das Modell hierdrauf.

Der Code und alle weiteren Informationen sind separat in [PVA2-CodingKata-2-Rock-Paper-Scissors.ipynb](PVA2-CodingKata-2-Rock-Paper-Scissors.ipynb) abgelegt.
